In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="flan_t5_symmetric_prompt_average_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

label_names = {0: "not_paraphrase", 1: "paraphrase"}
print(model_name)
print("vocab_size:", model.config.vocab_size)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


google/flan-t5-small
vocab_size: 32128


In [3]:
def get_single_token_id(text):
    candidates = [text, text.lower(), f" {text}", f" {text.lower()}"]
    for cand in candidates:
        ids = tokenizer.encode(cand, add_special_tokens=False)
        if len(ids) == 1:
            token = tokenizer.convert_ids_to_tokens(ids)[0]
            print({"requested": text, "matched_text": cand, "token_id": ids[0], "token": token})
            return ids[0]
    raise ValueError(f"Could not find a single-token form for {text!r}")

yes_token_id = get_single_token_id("yes")
no_token_id = get_single_token_id("no")

decoder_start_token_id = model.config.decoder_start_token_id
if decoder_start_token_id is None:
    decoder_start_token_id = tokenizer.pad_token_id

print("decoder_start_token_id:", decoder_start_token_id)
print("pad_token_id:", tokenizer.pad_token_id)


{'requested': 'yes', 'matched_text': 'yes', 'token_id': 4273, 'token': '▁yes'}
{'requested': 'no', 'matched_text': 'no', 'token_id': 150, 'token': '▁no'}
decoder_start_token_id: 0
pad_token_id: 0


In [4]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
def build_prompt(sentence_a, sentence_b):
    return (
        "Are the following two sentences paraphrases? Answer yes or no.\n"
        f"Sentence 1: {sentence_a}\n"
        f"Sentence 2: {sentence_b}"
    )

batch_size = 32
preds = []
forward_yes_logits = []
forward_no_logits = []
swapped_yes_logits = []
swapped_no_logits = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        prompts_forward = [build_prompt(a, b) for a, b in zip(batch_s1, batch_s2)]
        prompts_swapped = [build_prompt(b, a) for a, b in zip(batch_s1, batch_s2)]

        enc_forward = tokenizer(
            prompts_forward,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc_swapped = tokenizer(
            prompts_swapped,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )

        enc_forward = {k: v.to(device) for k, v in enc_forward.items()}
        enc_swapped = {k: v.to(device) for k, v in enc_swapped.items()}

        decoder_input_ids = torch.full(
            (len(batch_s1), 1),
            decoder_start_token_id,
            dtype=torch.long,
            device=device,
        )

        logits_forward = model(**enc_forward, decoder_input_ids=decoder_input_ids).logits[:, 0, :]
        logits_swapped = model(**enc_swapped, decoder_input_ids=decoder_input_ids).logits[:, 0, :]

        yes_forward = logits_forward[:, yes_token_id]
        no_forward = logits_forward[:, no_token_id]
        yes_swapped = logits_swapped[:, yes_token_id]
        no_swapped = logits_swapped[:, no_token_id]

        avg_yes_batch = (yes_forward + yes_swapped) / 2.0
        avg_no_batch = (no_forward + no_swapped) / 2.0
        batch_preds = (avg_yes_batch > avg_no_batch).long().cpu().numpy()

        preds.extend(batch_preds.tolist())
        forward_yes_logits.extend(yes_forward.cpu().numpy().tolist())
        forward_no_logits.extend(no_forward.cpu().numpy().tolist())
        swapped_yes_logits.extend(yes_swapped.cpu().numpy().tolist())
        swapped_no_logits.extend(no_swapped.cpu().numpy().tolist())

y_pred = np.array(preds)
forward_yes_logits = np.array(forward_yes_logits)
forward_no_logits = np.array(forward_no_logits)
swapped_yes_logits = np.array(swapped_yes_logits)
swapped_no_logits = np.array(swapped_no_logits)
avg_yes_logits = (forward_yes_logits + swapped_yes_logits) / 2.0
avg_no_logits = (forward_no_logits + swapped_no_logits) / 2.0

print("done")


  0%|          | 0/13 [00:00<?, ?it/s]

done


In [10]:

vault.create_record_list("google_flan_prediction_swapped", column_names=["prediction", "yes_logits", "no_logits", "swapped_yes_logits", "swapped_no_logits"])

for i in range(len(y_pred)):
    vault.append_record("google_flan_prediction_swapped", 
                        {
                            "prediction": y_pred[i],
                            "yes_logits": float(forward_yes_logits[i]),
                            "no_logits": float(forward_no_logits[i]),
                            "swapped_yes_logits": float(swapped_yes_logits[i]),
                            "swapped_no_logits": float(swapped_no_logits[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "Per-example prediction outputs from google/flan-t5-small on the GLUE MRPC validation set using a yes/no paraphrase prompt evaluated in both original and swapped sentence order. Each record is aligned to one input example from glue_mrpc_validation and contains: prediction (binary paraphrase decision after averaging forward and swapped logits), yes_logits (forward-order logit for \u201cyes\u201d), no_logits (forward-order logit for \u201cno\u201d), swapped_yes_logits (swapped-order logit for \u201cyes\u201d), and swapped_no_logits (swapped-order logit for \u201cno\u201d). This dataset is used as the intermediate model-output table in the workflow, supporting order-symmetric paraphrase prediction, error analysis, and computation of final evaluation metrics in the summary dataset."
embedding = get_embeddings(description)
vault.create_description("google_flan_prediction_swapped", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model predictions", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "google/flan-t5-small", "prompting": "yes/no question", "inference_method": "symmetric swapped-prompt average", "prediction_type": "binary classification", "output_columns": "prediction, yes_logits, no_logits, swapped_yes_logits, swapped_no_logits"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("google_flan_prediction_swapped", cat, embedding, prop)

NameError: name 'vault' is not defined

In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], zero_division=0)
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], zero_division=0))


{'accuracy': 0.7377450980392157, 'f1': 0.8065099457504521}
                precision    recall  f1-score   support

not_paraphrase       0.58      0.60      0.59       129
    paraphrase       0.81      0.80      0.81       279

      accuracy                           0.74       408
     macro avg       0.70      0.70      0.70       408
  weighted avg       0.74      0.74      0.74       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", label_names[int(y_pred[i])])
    print(
        {
            "forward_yes_logit": float(forward_yes_logits[i]),
            "forward_no_logit": float(forward_no_logits[i]),
            "swapped_yes_logit": float(swapped_yes_logits[i]),
            "swapped_no_logit": float(swapped_no_logits[i]),
            "avg_yes_logit": float(avg_yes_logits[i]),
            "avg_no_logit": float(avg_no_logits[i]),
        }
    )


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 label: not_paraphrase
{'forward_yes_logit': -1.8353147506713867, 'forward_no_logit': -1.884666919708252, 'swapped_yes_logit': -2.6327950954437256, 'swapped_no_logit': -2.5595462322235107, 'avg_yes_logit': -2.234054923057556, 'avg_no_logit': -2.2221065759658813}
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 label: not_paraphrase
{'forward_yes_logit': -2.955108404159546, 'forward_no_logit': -1.9274359941482544, 'swapped_yes_logit': -2.923551321029663, 'swapped_no_logit': -1.844859004020691, 'avg_yes_logit': -2.9393298625946045, 'avg_no_logit': -1.8861474990844727

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print(
        {
            "forward_yes_logit": float(forward_yes_logits[i]),
            "forward_no_logit": float(forward_no_logits[i]),
            "swapped_yes_logit": float(swapped_yes_logits[i]),
            "swapped_no_logit": float(swapped_no_logits[i]),
            "avg_yes_logit": float(avg_yes_logits[i]),
            "avg_no_logit": float(avg_no_logits[i]),
        }
    )


num_errors: 107
idx: 0
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0
{'forward_yes_logit': -1.8353147506713867, 'forward_no_logit': -1.884666919708252, 'swapped_yes_logit': -2.6327950954437256, 'swapped_no_logit': -2.5595462322235107, 'avg_yes_logit': -2.234054923057556, 'avg_no_logit': -2.2221065759658813}
idx: 9
sentence1: The results appear in the January issue of Cancer , an American Cancer Society journal , being published online today .
sentence2: The results appear in the January issue of Cancer , an American Cancer Society ( news - web sites ) journal , being published online Monday .
true: 1 pred: 0
{'forward_yes_logit': -3.6062777042388916, 'forward_no_logit': -2.8249077796936035, 'swapped_yes_logit': -2.6629745960235596, 'swapped_no_logit': -2.9104559421539307, 'avg_yes_logit': -3.1346261501312256, 'avg_no_logit': -2

In [9]:

vault.create_record_list("flan_t5_symmetric_prompt_average_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("flan_t5_symmetric_prompt_average_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "google_flan_prediction_swapped": [0, len(ds)]
                    })

summary

description = "Summary dataset for the FLAN-T5 symmetric-prompt MRPC evaluation run. It contains a single aggregate record describing model performance on the full glue_mrpc_validation split, computed from predictions stored in google_flan_prediction_swapped. The fields are: accuracy (float, overall classification accuracy), f1 (float, binary F1 score for paraphrase detection), and classification_report (string, the full scikit-learn classification report with precision, recall, F1, and support for both not_paraphrase and paraphrase classes). Its role in this workflow is to provide a compact experiment-level evaluation artifact that links the source validation examples and prediction outputs to final metrics for the notebook/process flan_t5_symmetric_prompt_average_mrpc."
embedding = get_embeddings(description)
vault.create_description("flan_t5_symmetric_prompt_average_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "google/flan-t5-small", "prompting_method": "symmetric prompt averaging", "prediction_target": "binary yes/no", "metrics": "accuracy,f1,classification_report", "input_items": "glue_mrpc_validation,google_flan_prediction_swapped"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_symmetric_prompt_average_mrpc_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'google/flan-t5-small',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.7377450980392157,
 'f1': 0.8065099457504521}

In [ ]:
description = "This notebook evaluates a zero-shot paraphrase detection workflow on the GLUE MRPC validation set using google/flan-t5-small. It frames MRPC as a yes/no text-to-text classification task by prompting the model with two sentences and asking whether they are paraphrases. To reduce order sensitivity, each sentence pair is scored twice\u2014once in the original order and once with the sentences swapped\u2014and the first-token logits for yes and no are averaged across both prompts to produce the final prediction. The notebook loads the validation data from TableVault, runs batched inference with Hugging Face Transformers on MPS or CPU, computes accuracy, F1, and a classification report, inspects example predictions and errors, and stores per-example predictions, logits, summary metrics, and metadata back into TableVault with embedding-based descriptions." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("flan_t5_symmetric_prompt_average_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "dataset": "glue/mrpc", "dataset_split": "validation", "model": "google/flan-t5-small", "model_family": "flan-t5", "inference_method": "zero-shot seq2seq first-token logit comparison", "prompt_strategy": "symmetric prompt averaging with sentence order swap", "labels": "yes/no mapped to paraphrase/not_paraphrase", "evaluation": "accuracy, f1-score, classification_report", "frameworks": "transformers, pytorch, datasets, sklearn", "storage": "tablevault", "database": "arangodb", "embedding_model": "text-embedding-3-large", "process_name": "flan_t5_symmetric_prompt_average_mrpc"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_symmetric_prompt_average_mrpc", cat, embedding, prop)